Finding GLADE+ galaxies inside the initial BAYESTAR localisation volume of GW170817.

The code was written by Mária Pálfi (marika97@student.elte.hu).

In [1]:
# importing useful packages:
import numpy as np

from astropy.coordinates import SkyCoord
from ligo.skymap.io import read_sky_map
from ligo.skymap.postprocess import crossmatch

from astroquery.vizier import VizierClass
from astroquery.vizier import Vizier

/media/SSD/anaconda_dir/anaconda3/lib/python3.11/site-packages/ligo/lw/lsctables.py:89: UserWarning: Wswiglal-redir-stdio:

SWIGLAL standard output/error redirection is enabled in IPython.
This may lead to performance penalties. To disable locally, use:

with lal.no_swig_redirect_standard_output_error():
    ...

To disable globally, use:

lal.swig_redirect_standard_output_error(False)

Note however that this will likely lead to error messages from
LAL functions being either misdirected or lost when called from
Jupyter notebooks.

To suppress this warning, use:

import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
import lal

  import lal


In [2]:
# finding GLADE+ catalogue in Vizier
catalog_list = Vizier.find_catalogs('GLADE+')
print({k:v.description for k,v in catalog_list.items()})

{'VII/275': 'GLADE catalog (Dalya+, 2016)', 'VII/281': 'GLADE v2.3 catalog (Dalya+, 2018)', 'VII/291': 'GLADE+ (Galaxy List for the Advanced Detector Era) (Dalya+, 2022)', 'J/ApJS/256/15': 'Revised GLADE & MANGROVE sample with HyperLEDA (Biteau, 2021)'}


In [3]:
# download GLADE+ coordinates
vizier = VizierClass(
    row_limit=-1,
    columns=['_GLADE_', '_RAJ2000', '_DEJ2000', '_dL'])
cat, = vizier.get_catalogs('VII/291')
cat.sort('GLADE_')  # sort catalog so that doctest output is stable

In [4]:
# creating a Skycoord object from the coordinates
coordinates = SkyCoord(cat['_RAJ2000'], cat['_DEJ2000'], cat['dL'])

In [5]:
# download initial BAYESTAR skymap
url = "https://dcc.ligo.org/public/0146/G1701985/001/bayestar.fits.gz"
skymap = read_sky_map(url, moc=True)

# crossmatch with GLADE+
result = crossmatch(skymap, coordinates)

In [6]:
print( 'Number of galaxies inside the 90% localisation volume:',  np.sum( result.searched_prob_vol < 0.9 ) )

Number of galaxies inside the 90% localisation volume: 58


In [7]:
# filtrating for the 90% localisation volume
cat_t = cat[result.searched_prob_vol < 0.9]

In [8]:
# saving to file the galaxy list
cat_t.write( '../total_host_list_vol_bayestar.csv', format='csv', overwrite = True )